# 01 · Define & Explore — KRAS biology, epitope/allele/nucleotide-state choice, binder metrics

**Standard slot:** *define & explore.* **For Project 08 this means:** understand why KRAS was
"undruggable", **choose the epitope (switch I/II or an allele pocket), the allele, and the
nucleotide state** on purpose, clean the KRAS G-domain, write down the binder metrics + cutoffs, and
run a deterministic **mock** mini-run (with a mock isoform panel) as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real binder campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

## Why KRAS is hard (and what we are designing)

KRAS is a small GTPase that cycles between a **GDP "off"** and a **GTP "on"** state; effectors (RAF,
PI3K) engage the "on" state. Activating codon-12 mutations (**G12C, G12D, G12V**) lock it on and drive
~25% of cancers. It was "undruggable" because the surface is small, smooth, charged, and pocket-poor.
The druggable handholds are the **switch I (~res 30–38)** and **switch II (~res 60–76)** regions, or an
**allele-specific** surface (e.g. the G12C cysteine). Two design decisions dominate everything:

1. **Nucleotide state** — the switch surface only exists in one state (GDP vs GTP/analog). Model the
   right one.
2. **Selectivity** — KRAS, HRAS, and NRAS are nearly identical across the switches, so a binder that
   hits KRAS usually hits all three. The centerpiece of this project is the **isoform-specificity
   panel** (notebook 04) and reasoning about **allele** selectivity.

## The binder metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–KRAS interface** (the key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |
| **isoform selectivity gap** | Å | KRAS pae minus best off-target (HRAS/NRAS) pae | measured selectivity |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** `pae_interaction` is the single most important binder metric — but a low value is
*confidence*, **not** affinity, and a positive selectivity gap is a *hypothesis* of selectivity. A
passing design is a **hypothesis** until SPR/BLI + the isoform panel.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Target prep + epitope/allele/nucleotide-state choice

The design target is the **KRAS G-domain** in a chosen **nucleotide state**, and the hotspots are the
KRAS residues of your chosen epitope — **switch I (~30–38)**, **switch II (~60–76)**, or an
**allele pocket**. Fetch the candidate structures with `data/download_data.py` (4OBE WT / 6OIM G12C —
**verify on RCSB**, and add HRAS/NRAS for the panel), isolate the KRAS chain, keep the bound
nucleotide + Mg²⁺ where the switch depends on them, remove waters, and read the hotspots off **that**
structure/state.

Below we just *declare* an EXAMPLE choice so the notebook runs end-to-end; **replace it with the
residues, allele, and state you derive** (numbering depends on the PDB you verify).

In [ ]:
import binder_tools as bt

TARGET = "KRAS"                 # cleaned KRAS G-domain (you produce this from 4OBE / 6OIM)
ALLELE = "G12C"                 # EXAMPLE allele you designed against — e.g. "WT", "G12C", "G12D"
NUCLEOTIDE_STATE = "GDP"        # EXAMPLE state — "GDP" ("off") or "GTP"/"GppNHp" ("on"); the switch surface depends on it
# EXAMPLE switch I/II hotspots — VERIFY/REPLACE from the actual KRAS structure + state (data/README.md).
# These are placeholders so the plumbing runs; real numbering depends on the PDB chain you clean.
HOTSPOTS = bt.parse_hotspots("A32,A35,A38,A60,A71")   # EXAMPLE_DATA: switch I (~30-38) + switch II (~60-76)
print("target          :", TARGET)
print("allele          :", ALLELE, " (EXAMPLE — record the allele you actually designed against)")
print("nucleotide state:", NUCLEOTIDE_STATE, " (EXAMPLE — the switch surface only exists in one state)")
print("hotspots        :", HOTSPOTS, " (EXAMPLE — replace with your verified switch I/II residues)")

## 2 · Mock hello-world: a tiny two-paradigm mini-run

`scripts/binder_tools.py` exposes both paradigms behind one API:
`generate_binders_bindcraft(...)` and `generate_binders_rfdiffusion(...)`, plus `af2_multimer(...)`
(the scorer) and `isoform_specificity(...)` / `specificity_panel(...)` (the KRAS-specific selectivity
helpers). The **mock** backend is deterministic and GPU-free so you can develop the plumbing.
**Never report mock numbers as real** — they are `SYNTHETIC` by construction, and there is no K_D anywhere.

In [ ]:
# A few designs from each paradigm, scored by mock AF2-Multimer. All numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=3, tool="mock", allele=ALLELE, nucleotide_state=NUCLEOTIDE_STATE)
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock", allele=ALLELE, nucleotide_state=NUCLEOTIDE_STATE)
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

d = bc[0]
print("example BindCraft design:")
print("  id    :", d.design_id)
print("  allele:", d.allele, " state:", d.nucleotide_state)
print("  len   :", d.length, "aa")
print("  seq   :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd,
      " sc =", d.shape_complementarity, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")

## 3 · Mock isoform-specificity panel (the KRAS-specific check)

The point of this project is not just "binds KRAS" but "binds KRAS and **not** HRAS/NRAS".
`specificity_panel()` re-scores a binder against each RAS isoform and computes the **selectivity gap**
(KRAS pae minus the best off-target pae; positive ⇒ some selectivity). Because the isoforms are nearly
identical across the switches, real selectivity is hard — here it is a deterministic SYNTHETIC proxy
you will replace with real AF2-Multimer runs vs HRAS/NRAS on Colab.

In [ ]:
for b in bc[:3]:
    panel = bt.specificity_panel(b, tool="mock")
    print(f"{b.design_id}: per-isoform pae = {panel['per_isoform']}  "
          f"selectivity_gap = {panel['selectivity_gap']}  selective = {panel['selective']}  (SYNTHETIC)")
print("\nDelta > 0 ⇒ KRAS scores better than the best off-target. On Colab, replace mock with real "
      "AF2-Multimer vs verified HRAS/NRAS structures. A pan-RAS binder (gap ~ 0) is a weaker result — report it.")

## 4 · Epitope coverage / effector-competition proxy

A switch-region binder could **block RAF/effector engagement** if it covers enough of the switch
footprint. `hotspot_overlap()` is a geometry proxy (fraction of chosen-epitope hotspots contacted) — a
teaching stand-in for the effector-competition assay in notebook 04. Higher ⇒ more likely to block
(not a guarantee).

In [ ]:
for b in bc[:3]:
    ov = bt.hotspot_overlap(b.contact_residues, HOTSPOTS)
    print(f"{b.design_id}: contacts {b.contact_residues} -> switch-footprint coverage = {ov} (SYNTHETIC)")

## Visualize a binder–KRAS complex (py3Dmol)

Use this to eyeball a predicted binder–KRAS complex once you have a real PDB (from AF2-Multimer) — and
to overlay KRAS/HRAS/NRAS at the switch regions when reasoning about selectivity.

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB):
# show_complex("results/af2/top_complex.pdb")
print("show_complex(pdb_path) ready.")

## D0 checklist
- [ ] KRAS accessions verified on RCSB (4OBE/6OIM are candidates); **HRAS/NRAS** added for the panel; KRAS chain identified.
- [ ] **Epitope, allele, and nucleotide state chosen and justified**; cleaned KRAS target + hotspot list (derived from that structure/state, not invented).
- [ ] One-paragraph definition of each binder metric **with** its "does not mean" note (incl. the selectivity gap).
- [ ] Reproduced mock mini-run (both paradigms) + mock isoform panel, with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria (incl. a selectivity target) + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the two-paradigm binder campaign at the chosen KRAS surface.